In [ ]:
#完整训练模块，运行直接训练
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from scipy.io import loadmat 
import numpy as np
import pandas as pd
import torch.nn as nn


# -----------------------------
# Dataset & DataLoaders
# -----------------------------
class MixtureOfGaussiansDataset(Dataset):
    def __init__(self, csv_file, mat_dir, mat_prefix='ps_train', mat_suffix='.mat', max_n=None, transform=None):
        """
        Args:
            csv_file (str): 输入参数CSV文件的路径。
            mat_dir (str): 包含density矩阵的.mat文件的目录。
            mat_prefix (str): .mat文件名的前缀（如 'ps_train', 'ps_valid', 'ps_test'）。
            mat_suffix (str): .mat文件名的后缀，通常为 '.mat'。
            max_n (int, required): 需要填充的密度矩阵的最大尺寸。
            transform (callable, optional): 应用于样本的可选转换。
        """
        self.data_frame = pd.read_csv(csv_file)
        self.mat_dir = mat_dir
        self.mat_prefix = mat_prefix
        self.mat_suffix = mat_suffix
        self.transform = transform

        # .mat 文件数量校验
        expected_mat_files = len(self.data_frame)
        actual_mat_files = len([f for f in os.listdir(mat_dir) if f.startswith(mat_prefix) and f.endswith(mat_suffix)])
        if actual_mat_files != expected_mat_files:
            raise ValueError(f"CSV文件中的样本数量 ({expected_mat_files}) 与.mat文件的数量 ({actual_mat_files}) 不匹配。")

        if max_n is None:
            raise ValueError("max_n必须指定，才能确保在所有数据集上找到最大尺寸。")
        self.max_n = max_n

        # 预加载并填充
        self.x_data, self.y_data = [], []
        for idx in range(expected_mat_files):
            params = self.data_frame.iloc[idx, :].values.astype(np.float32)
            x_tensor = torch.tensor(params, dtype=torch.float32)
            self.x_data.append(x_tensor)

            mat_filename = self.get_mat_filename(idx)
            mat_path = os.path.join(self.mat_dir, mat_filename)
            mat = loadmat(mat_path)
            density = mat.get('density')
            if density is None:
                raise KeyError(f"'density' 变量未在 {mat_path} 中找到。")
            density = density.astype(np.float32)
            density_tensor = torch.tensor(density)

            current_n = density_tensor.shape[0]
            pad_size = self.max_n - current_n
            if pad_size > 0:
                density_padded = F.pad(density_tensor, (0, pad_size, 0, pad_size), mode='constant', value=0)
            else:
                density_padded = density_tensor
            self.y_data.append(density_padded)

    def get_mat_filename(self, idx):
        """根据索引返回对应的.mat文件名（文件编号从1开始）。"""
        file_number = idx + 1
        return f'{self.mat_prefix}{{{file_number}}}{self.mat_suffix}'

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x = self.x_data[idx]
        y = self.y_data[idx]
        if self.transform:
            x = self.transform(x)
            y = self.transform(y)
        return x, y


def find_max_n(mat_dirs):
    """
    在给定的.mat文件目录中找到密度矩阵的最大尺寸。
    Args:
        mat_dirs (list of tuples): 每个元组包含 (mat_dir, mat_prefix, mat_suffix)。
    Returns:
        int: 最大密度矩阵尺寸。
    """
    max_n = 0
    for mat_dir, mat_prefix, mat_suffix in mat_dirs:
        for filename in os.listdir(mat_dir):
            if filename.startswith(mat_prefix) and filename.endswith(mat_suffix):
                mat_path = os.path.join(mat_dir, filename)
                mat = loadmat(mat_path)
                density = mat.get('density')
                if density is None:
                    raise KeyError(f"'density' variable not found in {mat_path}")
                current_n = density.shape[0]
                if current_n > max_n:
                    max_n = current_n
    return max_n


def prepare_dataloaders(csv_train, mat_train_dir, csv_valid, mat_valid_dir, csv_test, mat_test_dir,
                        batch_size=64, num_workers=4):
    """
    准备训练、验证和测试数据加载器。
    Returns:
        tuple: (train_loader, valid_loader, test_loader, max_n)
    """
    mat_dirs = [
        (mat_train_dir, 'ps_train', '.mat'),
        (mat_valid_dir, 'ps_valid', '.mat'),
        (mat_test_dir,  'ps_test',  '.mat')
    ]

    max_n = find_max_n(mat_dirs)
    print(f"全数据集中最大密度矩阵尺寸: {max_n}x{max_n}")

    train_dataset = MixtureOfGaussiansDataset(csv_file=csv_train, mat_dir=mat_train_dir,
                                              mat_prefix='ps_train', mat_suffix='.mat', max_n=max_n)
    valid_dataset = MixtureOfGaussiansDataset(csv_file=csv_valid, mat_dir=mat_valid_dir,
                                              mat_prefix='ps_valid', mat_suffix='.mat', max_n=max_n)
    test_dataset  = MixtureOfGaussiansDataset(csv_file=csv_test,  mat_dir=mat_test_dir,
                                              mat_prefix='ps_test',  mat_suffix='.mat', max_n=max_n)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    return train_loader, valid_loader, test_loader, max_n


# -----------------------------
# Loss: Hellinger distance (discrete) + moment penalty
# -----------------------------
class HellingerDistanceLoss(nn.Module):
    def __init__(self, grid_x, grid_y, grid_area, device='cpu'):
        """
        海灵格距离（离散格）损失 + 一阶矩惩罚。
        Args:
            grid_x (torch.Tensor): [n, n] 网格中心的x坐标。
            grid_y (torch.Tensor): [n, n] 网格中心的y坐标。
            grid_area (float): 网格面积（可选，若做积分一致性）。
            device (str): 计算设备。
        """
        super(HellingerDistanceLoss, self).__init__()
        self.grid_x = grid_x.to(device)  # [n, n]
        self.grid_y = grid_y.to(device)  # [n, n]
        self.grid_area = grid_area

    def forward(self, weights, means, covariances, true_density):
        """
        前向传播计算KL散度。

        Args:
            weights (torch.Tensor): [batch_size, num_components]
            means (torch.Tensor): [batch_size, num_components, 2]
            covariances (torch.Tensor): [batch_size, num_components, 2, 2]
            true_density (torch.Tensor): [batch_size, n, n]
        Returns:
            total_loss (scalar), HellingerDistance (per-sample)
        """
        batch_size, num_components = weights.shape
        n = true_density.shape[1]
        eps = 1e-8

        # 取出均值与协方差元素
        mu_x = means[:, :, 0].unsqueeze(-1).unsqueeze(-1)  # [B,K,1,1]
        mu_y = means[:, :, 1].unsqueeze(-1).unsqueeze(-1)  # [B,K,1,1]

        sigma_xx = covariances[:, :, 0, 0].unsqueeze(-1).unsqueeze(-1)
        sigma_xy = covariances[:, :, 0, 1].unsqueeze(-1).unsqueeze(-1)
        sigma_yy = covariances[:, :, 1, 1].unsqueeze(-1).unsqueeze(-1)

        det_sigma = sigma_xx * sigma_yy - sigma_xy ** 2
        det_sigma = torch.clamp(det_sigma, min=1e-12)
        inv_sigma_xx =  sigma_yy / det_sigma
        inv_sigma_xy = -sigma_xy / det_sigma
        inv_sigma_yy =  sigma_xx / det_sigma

        # 用广播避免 repeat 膨胀
        gx = self.grid_x.unsqueeze(0).unsqueeze(0)  # [1,1,n,n]
        gy = self.grid_y.unsqueeze(0).unsqueeze(0)  # [1,1,n,n]
        dx = gx - mu_x                               # [B,K,n,n]
        dy = gy - mu_y                               # [B,K,n,n]

        exponent = -0.5 * (inv_sigma_xx * dx ** 2 + 2 * inv_sigma_xy * dx * dy + inv_sigma_yy * dy ** 2)
        gaussian = (1.0 / (2 * torch.pi * torch.sqrt(det_sigma))) * torch.exp(exponent)  # [B,K,n,n]

        # 混合
        w = weights.unsqueeze(-1).unsqueeze(-1)   # [B,K,1,1]
        pred = torch.sum(w * gaussian, dim=1)     # [B,n,n]
        # 可选：pred = pred * self.grid_area（若做严格的“近似积分=1”）
        pred = pred + eps
        pred = pred / pred.sum(dim=[1, 2], keepdim=True)

        true = true_density + eps
        true = true / true.sum(dim=[1, 2], keepdim=True)

        # 一阶矩惩罚：log10 比值更稳健
        mean_pred_x = torch.sum(pred * self.grid_x, dim=[1, 2])
        mean_true_x = torch.sum(true * self.grid_x, dim=[1, 2])
        mean_pred_y = torch.sum(pred * self.grid_y, dim=[1, 2])
        mean_true_y = torch.sum(true * self.grid_y, dim=[1, 2])

        lamda = 0.02
        ratio_x = (mean_pred_x + 1e-6) / (mean_true_x + 1e-6)
        ratio_y = (mean_pred_y + 1e-6) / (mean_true_y + 1e-6)
        penalty = lamda * (torch.log10(ratio_x) ** 2 + torch.log10(ratio_y) ** 2)  # [B]

        # Hellinger
        sqrt_pred = torch.sqrt(pred.view(batch_size, -1))
        sqrt_true = torch.sqrt(true.view(batch_size, -1))
        hell2 = 0.5 * torch.sum((sqrt_pred - sqrt_true) ** 2, dim=1)  # [B]
        hell  = torch.sqrt(torch.clamp(hell2, min=0.0) + 1e-12)       # [B]

        total = hell.mean() + penalty.mean()
        return total, hell


# -----------------------------
# Model
# -----------------------------
class MixtureOfGaussiansModel(nn.Module):
    def __init__(self, input_dim=4, num_components=4, hidden_dims=None):
        """
        支持动态隐藏层数的混合高斯模型。
        """
        super(MixtureOfGaussiansModel, self).__init__()
        if hidden_dims is None or len(hidden_dims) == 0:
            raise ValueError("hidden_dims 不能为空，且需至少包含一层。")

        self.input_dim = input_dim
        self.num_components = num_components
        self.hidden_dims = hidden_dims

        # 动态构建隐藏层
        self.hidden_layers = nn.ModuleList()
        self.hidden_layers.append(nn.Linear(input_dim, hidden_dims[0]))
        for i in range(1, len(hidden_dims)):
            self.hidden_layers.append(nn.Linear(hidden_dims[i-1], hidden_dims[i]))

        # 输出层：每个组件 6 个参数 (w, mu_x, mu_y, s_xx, alpha, s_yy)
        self.output_layer = nn.Linear(hidden_dims[-1], num_components * 6)

    def forward(self, x):
        """
        Returns:
            weights: [B, K]
            means: [B, K, 2]
            covariances: [B, K, 2, 2]
        """
        batch_size = x.shape[0]

        # 隐藏层 + ReLU
        for layer in self.hidden_layers:
            x = F.relu(layer(x))

        # 输出并重排
        output = self.output_layer(x).view(batch_size, self.num_components, 6)

        # 权重
        weights = F.softmax(output[:, :, 0], dim=1)  # [B,K]

        # 均值
        means = output[:, :, 1:3]  # [B,K,2]

        # 协方差参数化（数值稳定）
        eps = 1e-6
        s_xx  = F.softplus(output[:, :, 3]) + eps
        alpha = output[:, :, 4]
        rho   = 2 * torch.sigmoid(alpha) - 1
        rho   = torch.clamp(rho, -1 + 1e-4, 1 - 1e-4)
        s_yy  = F.softplus(output[:, :, 5]) + eps
        sigma_xy = rho * torch.sqrt(s_xx * s_yy)  # [B,K]

        covariances = torch.zeros(batch_size, self.num_components, 2, 2, device=x.device)
        covariances[:, :, 0, 0] = s_xx
        covariances[:, :, 0, 1] = sigma_xy
        covariances[:, :, 1, 0] = sigma_xy
        covariances[:, :, 1, 1] = s_yy

        return weights, means, covariances


# -----------------------------
# Train / Eval
# -----------------------------
def train_model(csv_train, mat_train_dir, csv_valid, mat_valid_dir, csv_test, mat_test_dir,
                num_epochs, batch_size, learning_rate, min_learning_rate, num_workers,
                hidden_dims):
    """
    训练神经网络模型。
    仅需在 __main__ 中修改 hidden_dims=[...]，此处会自动生效并统一到保存/加载文件名。
    """
    # 打印结构
    print(f"[INFO] hidden_dims = {hidden_dims}")

    # DataLoaders
    train_loader, valid_loader, test_loader, max_n = prepare_dataloaders(
        csv_train=csv_train,
        mat_train_dir=mat_train_dir,
        csv_valid=csv_valid,
        mat_valid_dir=mat_valid_dir,
        csv_test=csv_test,
        mat_test_dir=mat_test_dir,
        batch_size=batch_size,
        num_workers=num_workers
    )

    # 网格 & 设备
    lim = max_n * 0.05
    edges = torch.linspace(0.025, lim - 0.025, steps=max_n)
    grid_x, grid_y = torch.meshgrid(edges, edges, indexing='ij')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    grid_x = grid_x.to(device)
    grid_y = grid_y.to(device)
    grid_area = 0.05 * 0.05

    # 模型
    model = MixtureOfGaussiansModel(input_dim=4, num_components=3, hidden_dims=hidden_dims).to(device)

    # 优化与调度
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience=10, min_lr=min_learning_rate
    )

    # 损失
    loss_fn = HellingerDistanceLoss(grid_x, grid_y, grid_area, device=device).to(device)

    # 唯一 ckpt 名称（随 hidden_dims 变化）
    dims_tag = "-".join(map(str, hidden_dims))
    ckpt_path = f'best_model_{dims_tag}.pth'

    # 训练循环
    best_val_loss = float('inf')
    patience = 50
    trigger_times = 0

    for epoch in range(num_epochs):
        model.train()
        train_losses, train_hellinger_distances = [], []
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            weights, means, covariances = model(batch_x)
            total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
            total_loss.backward()
            optimizer.step()

            train_losses.append(total_loss.item())
            train_hellinger_distances.append(hellinger_distance.mean().item())

        avg_train_loss = np.mean(train_losses)
        avg_train_hellinger = np.mean(train_hellinger_distances)

        # 验证
        model.eval()
        val_losses, val_hellinger_distances = [], []
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                weights, means, covariances = model(batch_x)
                total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
                val_losses.append(total_loss.item())
                val_hellinger_distances.append(hellinger_distance.mean().item())

        avg_val_loss = np.mean(val_losses)
        avg_val_hellinger = np.mean(val_hellinger_distances)

        # 调度
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {avg_train_loss:.6f}, Train Hellinger: {avg_train_hellinger:.6f}, "
              f"Val Loss: {avg_val_loss:.6f}, Val Hellinger: {avg_val_hellinger:.6f}, "
              f"Current LR: {current_lr:.6f}")

        # 早停 & 保存
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"保存最佳模型到 {ckpt_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            print(f"验证损失未改善轮数: {trigger_times}")
            if trigger_times >= patience:
                print("达到早停条件，停止训练。")
                break

    # 测试：加载最佳权重（与 hidden_dims 一一对应）
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)

    model.eval()
    test_losses, test_hellinger_distances = [], []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            weights, means, covariances = model(batch_x)
            total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
            test_losses.append(total_loss.item())
            test_hellinger_distances.append(hellinger_distance.mean().item())

    avg_test_loss = np.mean(test_losses)
    avg_test_hellinger = np.mean(test_hellinger_distances)
    print(f"测试集平均 Total Loss: {avg_test_loss:.6f}, 平均 Hellinger Distance: {avg_test_hellinger:.6f}")



if __name__ == "__main__":
    # 数据路径
    csv_train    = 'ps_train.csv'
    mat_train_dir= 'density_results_ps_train_n_cells=50000_edges=0.05'
    csv_valid    = 'ps_valid.csv'
    mat_valid_dir= 'density_results_ps_valid_n_cells=50000_edges=0.05'
    csv_test     = 'ps_test.csv'
    mat_test_dir = 'density_results_ps_test_n_cells=50000_edges=0.05'

    # 训练参数
    num_epochs = 2000
    batch_size = 32
    learning_rate = 1e-3
    num_workers = 0
    min_learning_rate = learning_rate / 32

    # —— 只需要在这里改 —— #
    hidden_dims = [256, 512, 256]   # 比如改成 [256, 256] 或 [1024,1024,1024] 等

    # 启动训练（hidden_dims 传入一次，其他模块自动使用）
    train_model(
        csv_train=csv_train,
        mat_train_dir=mat_train_dir,
        csv_valid=csv_valid,
        mat_valid_dir=mat_valid_dir,
        csv_test=csv_test,
        mat_test_dir=mat_test_dir,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        num_workers=num_workers,
        min_learning_rate=min_learning_rate,
        hidden_dims=hidden_dims
    )

In [ ]:
#训练模块代码，不带主程序
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from scipy.io import loadmat 
import numpy as np
import pandas as pd
import torch.nn as nn


# -----------------------------
# Dataset & DataLoaders
# -----------------------------
class MixtureOfGaussiansDataset(Dataset):
    def __init__(self, csv_file, mat_dir, mat_prefix='ps_train', mat_suffix='.mat', max_n=None, transform=None):
        """
        Args:
            csv_file (str): 输入参数CSV文件的路径。
            mat_dir (str): 包含density矩阵的.mat文件的目录。
            mat_prefix (str): .mat文件名的前缀（如 'ps_train', 'ps_valid', 'ps_test'）。
            mat_suffix (str): .mat文件名的后缀，通常为 '.mat'。
            max_n (int, required): 需要填充的密度矩阵的最大尺寸。
            transform (callable, optional): 应用于样本的可选转换。
        """
        self.data_frame = pd.read_csv(csv_file)
        self.mat_dir = mat_dir
        self.mat_prefix = mat_prefix
        self.mat_suffix = mat_suffix
        self.transform = transform

        # .mat 文件数量校验
        expected_mat_files = len(self.data_frame)
        actual_mat_files = len([f for f in os.listdir(mat_dir) if f.startswith(mat_prefix) and f.endswith(mat_suffix)])
        if actual_mat_files != expected_mat_files:
            raise ValueError(f"CSV文件中的样本数量 ({expected_mat_files}) 与.mat文件的数量 ({actual_mat_files}) 不匹配。")

        if max_n is None:
            raise ValueError("max_n必须指定，才能确保在所有数据集上找到最大尺寸。")
        self.max_n = max_n

        # 预加载并填充
        self.x_data, self.y_data = [], []
        for idx in range(expected_mat_files):
            params = self.data_frame.iloc[idx, :].values.astype(np.float32)
            x_tensor = torch.tensor(params, dtype=torch.float32)
            self.x_data.append(x_tensor)

            mat_filename = self.get_mat_filename(idx)
            mat_path = os.path.join(self.mat_dir, mat_filename)
            mat = loadmat(mat_path)
            density = mat.get('density')
            if density is None:
                raise KeyError(f"'density' 变量未在 {mat_path} 中找到。")
            density = density.astype(np.float32)
            density_tensor = torch.tensor(density)

            current_n = density_tensor.shape[0]
            pad_size = self.max_n - current_n
            if pad_size > 0:
                density_padded = F.pad(density_tensor, (0, pad_size, 0, pad_size), mode='constant', value=0)
            else:
                density_padded = density_tensor
            self.y_data.append(density_padded)

    def get_mat_filename(self, idx):
        """根据索引返回对应的.mat文件名（文件编号从1开始）。"""
        file_number = idx + 1
        return f'{self.mat_prefix}{{{file_number}}}{self.mat_suffix}'

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x = self.x_data[idx]
        y = self.y_data[idx]
        if self.transform:
            x = self.transform(x)
            y = self.transform(y)
        return x, y


def find_max_n(mat_dirs):
    """
    在给定的.mat文件目录中找到密度矩阵的最大尺寸。
    Args:
        mat_dirs (list of tuples): 每个元组包含 (mat_dir, mat_prefix, mat_suffix)。
    Returns:
        int: 最大密度矩阵尺寸。
    """
    max_n = 0
    for mat_dir, mat_prefix, mat_suffix in mat_dirs:
        for filename in os.listdir(mat_dir):
            if filename.startswith(mat_prefix) and filename.endswith(mat_suffix):
                mat_path = os.path.join(mat_dir, filename)
                mat = loadmat(mat_path)
                density = mat.get('density')
                if density is None:
                    raise KeyError(f"'density' variable not found in {mat_path}")
                current_n = density.shape[0]
                if current_n > max_n:
                    max_n = current_n
    return max_n


def prepare_dataloaders(csv_train, mat_train_dir, csv_valid, mat_valid_dir, csv_test, mat_test_dir,
                        batch_size=64, num_workers=4):
    """
    准备训练、验证和测试数据加载器。
    Returns:
        tuple: (train_loader, valid_loader, test_loader, max_n)
    """
    mat_dirs = [
        (mat_train_dir, 'ps_train', '.mat'),
        (mat_valid_dir, 'ps_valid', '.mat'),
        (mat_test_dir,  'ps_test',  '.mat')
    ]

    max_n = find_max_n(mat_dirs)
    print(f"全数据集中最大密度矩阵尺寸: {max_n}x{max_n}")

    train_dataset = MixtureOfGaussiansDataset(csv_file=csv_train, mat_dir=mat_train_dir,
                                              mat_prefix='ps_train', mat_suffix='.mat', max_n=max_n)
    valid_dataset = MixtureOfGaussiansDataset(csv_file=csv_valid, mat_dir=mat_valid_dir,
                                              mat_prefix='ps_valid', mat_suffix='.mat', max_n=max_n)
    test_dataset  = MixtureOfGaussiansDataset(csv_file=csv_test,  mat_dir=mat_test_dir,
                                              mat_prefix='ps_test',  mat_suffix='.mat', max_n=max_n)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    return train_loader, valid_loader, test_loader, max_n


# -----------------------------
# Loss: Hellinger distance (discrete) + moment penalty
# -----------------------------
class HellingerDistanceLoss(nn.Module):
    def __init__(self, grid_x, grid_y, grid_area, device='cpu'):
        """
        海灵格距离（离散格）损失 + 一阶矩惩罚。
        Args:
            grid_x (torch.Tensor): [n, n] 网格中心的x坐标。
            grid_y (torch.Tensor): [n, n] 网格中心的y坐标。
            grid_area (float): 网格面积（可选，若做积分一致性）。
            device (str): 计算设备。
        """
        super(HellingerDistanceLoss, self).__init__()
        self.grid_x = grid_x.to(device)  # [n, n]
        self.grid_y = grid_y.to(device)  # [n, n]
        self.grid_area = grid_area

    def forward(self, weights, means, covariances, true_density):
        """
        前向传播计算KL散度。

        Args:
            weights (torch.Tensor): [batch_size, num_components]
            means (torch.Tensor): [batch_size, num_components, 2]
            covariances (torch.Tensor): [batch_size, num_components, 2, 2]
            true_density (torch.Tensor): [batch_size, n, n]
        Returns:
            total_loss (scalar), HellingerDistance (per-sample)
        """
        batch_size, num_components = weights.shape
        n = true_density.shape[1]
        eps = 1e-8

        # 取出均值与协方差元素
        mu_x = means[:, :, 0].unsqueeze(-1).unsqueeze(-1)  # [B,K,1,1]
        mu_y = means[:, :, 1].unsqueeze(-1).unsqueeze(-1)  # [B,K,1,1]

        sigma_xx = covariances[:, :, 0, 0].unsqueeze(-1).unsqueeze(-1)
        sigma_xy = covariances[:, :, 0, 1].unsqueeze(-1).unsqueeze(-1)
        sigma_yy = covariances[:, :, 1, 1].unsqueeze(-1).unsqueeze(-1)

        det_sigma = sigma_xx * sigma_yy - sigma_xy ** 2
        det_sigma = torch.clamp(det_sigma, min=1e-12)
        inv_sigma_xx =  sigma_yy / det_sigma
        inv_sigma_xy = -sigma_xy / det_sigma
        inv_sigma_yy =  sigma_xx / det_sigma

        # 用广播避免 repeat 膨胀
        gx = self.grid_x.unsqueeze(0).unsqueeze(0)  # [1,1,n,n]
        gy = self.grid_y.unsqueeze(0).unsqueeze(0)  # [1,1,n,n]
        dx = gx - mu_x                               # [B,K,n,n]
        dy = gy - mu_y                               # [B,K,n,n]

        exponent = -0.5 * (inv_sigma_xx * dx ** 2 + 2 * inv_sigma_xy * dx * dy + inv_sigma_yy * dy ** 2)
        gaussian = (1.0 / (2 * torch.pi * torch.sqrt(det_sigma))) * torch.exp(exponent)  # [B,K,n,n]

        # 混合
        w = weights.unsqueeze(-1).unsqueeze(-1)   # [B,K,1,1]
        pred = torch.sum(w * gaussian, dim=1)     # [B,n,n]
        # 可选：pred = pred * self.grid_area（若做严格的“近似积分=1”）
        pred = pred + eps
        pred = pred / pred.sum(dim=[1, 2], keepdim=True)

        true = true_density + eps
        true = true / true.sum(dim=[1, 2], keepdim=True)

        # 一阶矩惩罚：log10 比值更稳健
        mean_pred_x = torch.sum(pred * self.grid_x, dim=[1, 2])
        mean_true_x = torch.sum(true * self.grid_x, dim=[1, 2])
        mean_pred_y = torch.sum(pred * self.grid_y, dim=[1, 2])
        mean_true_y = torch.sum(true * self.grid_y, dim=[1, 2])

        lamda = 0.02
        ratio_x = (mean_pred_x + 1e-6) / (mean_true_x + 1e-6)
        ratio_y = (mean_pred_y + 1e-6) / (mean_true_y + 1e-6)
        penalty = lamda * (torch.log10(ratio_x) ** 2 + torch.log10(ratio_y) ** 2)  # [B]

        # Hellinger
        sqrt_pred = torch.sqrt(pred.view(batch_size, -1))
        sqrt_true = torch.sqrt(true.view(batch_size, -1))
        hell2 = 0.5 * torch.sum((sqrt_pred - sqrt_true) ** 2, dim=1)  # [B]
        hell  = torch.sqrt(torch.clamp(hell2, min=0.0) + 1e-12)       # [B]

        total = hell.mean() + penalty.mean()
        return total, hell


# -----------------------------
# Model
# -----------------------------
class MixtureOfGaussiansModel(nn.Module):
    def __init__(self, input_dim=4, num_components=4, hidden_dims=None):
        """
        支持动态隐藏层数的混合高斯模型。
        """
        super(MixtureOfGaussiansModel, self).__init__()
        if hidden_dims is None or len(hidden_dims) == 0:
            raise ValueError("hidden_dims 不能为空，且需至少包含一层。")

        self.input_dim = input_dim
        self.num_components = num_components
        self.hidden_dims = hidden_dims

        # 动态构建隐藏层
        self.hidden_layers = nn.ModuleList()
        self.hidden_layers.append(nn.Linear(input_dim, hidden_dims[0]))
        for i in range(1, len(hidden_dims)):
            self.hidden_layers.append(nn.Linear(hidden_dims[i-1], hidden_dims[i]))

        # 输出层：每个组件 6 个参数 (w, mu_x, mu_y, s_xx, alpha, s_yy)
        self.output_layer = nn.Linear(hidden_dims[-1], num_components * 6)

    def forward(self, x):
        """
        Returns:
            weights: [B, K]
            means: [B, K, 2]
            covariances: [B, K, 2, 2]
        """
        batch_size = x.shape[0]

        # 隐藏层 + ReLU
        for layer in self.hidden_layers:
            x = F.relu(layer(x))

        # 输出并重排
        output = self.output_layer(x).view(batch_size, self.num_components, 6)

        # 权重
        weights = F.softmax(output[:, :, 0], dim=1)  # [B,K]

        # 均值
        means = output[:, :, 1:3]  # [B,K,2]

        # 协方差参数化（数值稳定）
        eps = 1e-6
        s_xx  = F.softplus(output[:, :, 3]) + eps
        alpha = output[:, :, 4]
        rho   = 2 * torch.sigmoid(alpha) - 1
        rho   = torch.clamp(rho, -1 + 1e-4, 1 - 1e-4)
        s_yy  = F.softplus(output[:, :, 5]) + eps
        sigma_xy = rho * torch.sqrt(s_xx * s_yy)  # [B,K]

        covariances = torch.zeros(batch_size, self.num_components, 2, 2, device=x.device)
        covariances[:, :, 0, 0] = s_xx
        covariances[:, :, 0, 1] = sigma_xy
        covariances[:, :, 1, 0] = sigma_xy
        covariances[:, :, 1, 1] = s_yy

        return weights, means, covariances


# -----------------------------
# Train / Eval
# -----------------------------
def train_model(csv_train, mat_train_dir, csv_valid, mat_valid_dir, csv_test, mat_test_dir,
                num_epochs, batch_size, learning_rate, min_learning_rate, num_workers,
                hidden_dims):
    """
    训练神经网络模型。
    仅需在 __main__ 中修改 hidden_dims=[...]，此处会自动生效并统一到保存/加载文件名。
    """
    # 打印结构
    print(f"[INFO] hidden_dims = {hidden_dims}")

    # DataLoaders
    train_loader, valid_loader, test_loader, max_n = prepare_dataloaders(
        csv_train=csv_train,
        mat_train_dir=mat_train_dir,
        csv_valid=csv_valid,
        mat_valid_dir=mat_valid_dir,
        csv_test=csv_test,
        mat_test_dir=mat_test_dir,
        batch_size=batch_size,
        num_workers=num_workers
    )

    # 网格 & 设备
    lim = max_n * 0.05
    edges = torch.linspace(0.025, lim - 0.025, steps=max_n)
    grid_x, grid_y = torch.meshgrid(edges, edges, indexing='ij')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    grid_x = grid_x.to(device)
    grid_y = grid_y.to(device)
    grid_area = 0.05 * 0.05

    # 模型
    model = MixtureOfGaussiansModel(input_dim=4, num_components=4, hidden_dims=hidden_dims).to(device)

    # 优化与调度
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience=10, min_lr=min_learning_rate
    )

    # 损失
    loss_fn = HellingerDistanceLoss(grid_x, grid_y, grid_area, device=device).to(device)

    # 唯一 ckpt 名称（随 hidden_dims 变化）
    dims_tag = "-".join(map(str, hidden_dims))
    ckpt_path = f'best_model_{dims_tag}.pth'

    # 训练循环
    best_val_loss = float('inf')
    patience = 50
    trigger_times = 0

    for epoch in range(num_epochs):
        model.train()
        train_losses, train_hellinger_distances = [], []
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            weights, means, covariances = model(batch_x)
            total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
            total_loss.backward()
            optimizer.step()

            train_losses.append(total_loss.item())
            train_hellinger_distances.append(hellinger_distance.mean().item())

        avg_train_loss = np.mean(train_losses)
        avg_train_hellinger = np.mean(train_hellinger_distances)

        # 验证
        model.eval()
        val_losses, val_hellinger_distances = [], []
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                weights, means, covariances = model(batch_x)
                total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
                val_losses.append(total_loss.item())
                val_hellinger_distances.append(hellinger_distance.mean().item())

        avg_val_loss = np.mean(val_losses)
        avg_val_hellinger = np.mean(val_hellinger_distances)

        # 调度
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {avg_train_loss:.6f}, Train Hellinger: {avg_train_hellinger:.6f}, "
              f"Val Loss: {avg_val_loss:.6f}, Val Hellinger: {avg_val_hellinger:.6f}, "
              f"Current LR: {current_lr:.6f}")

        # 早停 & 保存
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"保存最佳模型到 {ckpt_path}")
            trigger_times = 0
        else:
            trigger_times += 1
            print(f"验证损失未改善轮数: {trigger_times}")
            if trigger_times >= patience:
                print("达到早停条件，停止训练。")
                break

    # 测试：加载最佳权重（与 hidden_dims 一一对应）
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)

    model.eval()
    test_losses, test_hellinger_distances = [], []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            weights, means, covariances = model(batch_x)
            total_loss, hellinger_distance = loss_fn(weights, means, covariances, batch_y)
            test_losses.append(total_loss.item())
            test_hellinger_distances.append(hellinger_distance.mean().item())

    avg_test_loss = np.mean(test_losses)
    avg_test_hellinger = np.mean(test_hellinger_distances)
    print(f"测试集平均 Total Loss: {avg_test_loss:.6f}, 平均 Hellinger Distance: {avg_test_hellinger:.6f}")


In [ ]:
#训练主程序
if __name__ == "__main__":
    # 数据路径
    csv_train    = 'ps_train.csv'
    mat_train_dir= 'density_results_ps_train_n_cells=50000_edges=0.05'
    csv_valid    = 'ps_valid.csv'
    mat_valid_dir= 'density_results_ps_valid_n_cells=50000_edges=0.05'
    csv_test     = 'ps_test.csv'
    mat_test_dir = 'density_results_ps_test_n_cells=50000_edges=0.05'

    # 训练参数
    num_epochs = 2000
    batch_size = 32
    learning_rate = 1e-3
    num_workers = 0
    min_learning_rate = learning_rate / 32

    # —— 只需要在这里改 —— #
    hidden_dims = [256, 512, 256]   # 比如改成 [256, 256] 或 [1024,1024,1024] 等

    # 启动训练（hidden_dims 传入一次，其他模块自动使用）
    train_model(
        csv_train=csv_train,
        mat_train_dir=mat_train_dir,
        csv_valid=csv_valid,
        mat_valid_dir=mat_valid_dir,
        csv_test=csv_test,
        mat_test_dir=mat_test_dir,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        num_workers=num_workers,
        min_learning_rate=min_learning_rate,
        hidden_dims=hidden_dims
    )

In [ ]:
#二维画图
# 1. 加载模型
model = MixtureOfGaussiansModel(input_dim=4, num_components=4, hidden_dims=[256, 512, 256])
model.load_state_dict(torch.load('best_model_256-512-256.pth', map_location='cpu',weights_only=True))
model.eval()

# 2. 输入你的一组参数（例如 4 维向量）
#input_params = [1.3048828125,	1.1548828125,	0.7759765625,	0.364501953125]   #train1
#input_params = [0.4048828125,	1.4548828125,	0.6759765625,	0.489501953125] #3
#input_params = [0.4423828125,	0.8923828125,	0.9384765625,	0.005126953125] #4
#input_params = [1.3423828125,	0.5923828125,	0.8384765625,	0.130126953125] #6
#input_params = [3.375,	2.05,	0.975,	0.825,	0.275] #13
input_params = [0.700195313,	0.765820313,	0.893164063,	0.022705078]#test 128
#input_params = [0.557226563,	1.307226563,	0.687695313,	0.004150391]#test 363
#input_params = [1.321289063,	0.486914063,	0.961132813,	0.435791016]#test 198




x = torch.tensor([input_params], dtype=torch.float32)

# 3. 得到预测输出
with torch.no_grad():
    weights, means, covariances = model(x)

# 4. 打印结果
print("预测的混合权重：", weights.numpy())
print("预测的均值：", means.numpy())
print("预测的协方差矩阵：", covariances.numpy())

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import torch

# 假设你已经得到了以下张量（从模型输出中提取）：
# weights: [num_components]
# means: [num_components, 2]
# covariances: [num_components, 2, 2]
# 例如下面这三行来自模型预测
weights = weights[0].cpu().numpy()          # shape: (3,)
means = means[0].cpu().numpy()              # shape: (3, 2)
covariances = covariances[0].cpu().numpy()  # shape: (3, 2, 2)

# 创建网格用于绘图
x = np.linspace(0, 2, 200)  # 你可以根据实际分布范围修改
y = np.linspace(0, 2, 200)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))  # shape: [200, 200, 2]

# 计算混合高斯分布的密度
Z = np.zeros(X.shape)
for i in range(len(weights)):
    rv = multivariate_normal(mean=means[i], cov=covariances[i])
    Z += weights[i] * rv.pdf(pos)
# 添加极小值避免零概率问题
#Z += 1e-7  # 给每个元素加上10^-7
# 归一化处理
#Z /= Z.sum()  # 确保所有概率之和为1
# 添加 X 轴翻转

# 绘图
plt.figure(figsize=(6, 5))
#plt.contourf(X, Y, -np.log(Z), levels=50, cmap='viridis')  # 也可以使用 plt.imshow(Z) 或 plt.contour
plt.contourf(X, Y, -np.log(np.maximum(Z, 1e-10)), levels=50, cmap='jet')  # 也可以使用 plt.imshow(Z) 或 plt.contour
#plt.contourf(X, Y, Z, levels=50, cmap='viridis')  # 也可以使用 plt.imshow(Z) 或 plt.contour
#plt.colorbar(label='Density')
plt.colorbar(label='Landscape')
#plt.scatter(means[:, 0], means[:, 1], c='red', marker='x', label='Means')
plt.title("2D Landscape")
plt.xlabel("X")
plt.ylabel("Y")
#plt.legend()
plt.tight_layout()
#plt.gca().invert_xaxis()  # 这是关键的一行，翻转 X 轴
plt.show()